# Homework 05: Transformers
**Submission Deadline: July 17, 2026, Friday, 9:30**

---

> **Machine Learning - Foundations and Algorithms** in Summer Semester 2026
>
> - Prof. Gerhard Neumann
> - Instructors for this exercise: Florian Seligmann (florian.seligmann@kit.edu) and Emiliyan Gospodinov (emiliyan.gospodinov@kit.edu)
>
> For general questions about the exercises, please post in the Ilias forum so we can answer them once for all students. Only use email for individual questions whose answers are not relevant to all students.

---


## Homework Info

This homework is all about transformers.
You will implement a full encoder-decoder transformer from scratch, following the "Attention is All You Need" paper, and then train it on a German-to-English translation task.

The homework contains coding tasks, multiple-choice questions, and an optional symbolic derivation. **Only coding and multiple-choice tasks are graded.** Tasks marked with "(Not Graded)" are included to strengthen your understanding, prepare you for the exam, and support discussion in the exercise session.

**>>> ADDITIONAL HINTS FOR THIS EXERCISE; PLEASE READ <<<**
- This homework contains quite a lot of explanatory text. We recommend to read this text as it helps you to better understand how a transformer works and what you need to implement.
- Some of the tasks build on top of previous tasks. In case you didn't manage to implement the previous tasks, you have the option to swap in PyTorch's built-in implementations. For grading, we'll replace your code by ground-truth implementations, so don't worry if you don't manage to implement something; you'll still receive points.
- In the end, you will have the option to train your transformer. This takes quite a lot of compute resources (multiple minutes on a very strong consumer GPU, potentially hours on a CPU). We do not grade based on the training result, so if you don't want to, you don't necessarily need to train for all epochs.

## Group Submission

- Exercise sheets may be submitted in **groups of up to 3 students**.
- **Every group member must upload the edited notebook to Ilias.** It is not sufficient for only one member to submit. If you forget to upload but at least one other group member submitted the group solution, you receive two thirds of the points the first time this happens and zero points if it happens again.
- **The submission deadline is strict.** Late submissions receive zero points, and submissions by email are not accepted.
- You may join a new group if your previous group dissolves. Each group must submit its own work; duplicate submissions across groups will be checked.
- You can use the team-building forum on Ilias to find group members.

Enter the u-identifiers of your group members in the next cell. Leave unused fields empty.


U-identifiers of group members:

Member 1:

Member 2:

Member 3:


## Auto-grading

We use an auto-grading system that checks graded answers with visible and hidden tests. Hidden tests determine the final points for coding and multiple-choice tasks.

To ensure auto-grading works correctly:

- Keep the filename `ex_05_transformers.ipynb`.
- Upload only the edited notebook to Ilias as a single raw `.ipynb` file. Do not upload a zip archive or any data files. In particular, **do not upload any saved models**! They are too large for Ilias.
- Before submission, restart the kernel and run the notebook from beginning to end. It must complete without errors.
- Enter solutions only in the designated answer and code cells.
- Do not delete, add, reorder, or rename provided cells, functions, parameters, return values, or answer variables.
- Visible asserts help identify common mistakes. Passing them is necessary but not sufficient because additional hidden tests are used for grading.

**If you fail to comply with these rules, your submission may not be graded and you may receive no points for this exercise.**


## 1) Transformer from Scratch (20P)

In this task, you will implement a transformer model from scratch, following the original paper ["Attention is All You Need"](https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf).
Then you will train it as a German to English translator.
The model will be a small, encoder-decoder transformer.

First, some imports:

In [ ]:
##### DO NOT CHANGE #####
import math

import torch
import torch.nn as nn

##### DO NOT CHANGE #####

We will use the Multi30K dataset which contains 30k simple German-English sentence pairs. Let's firstly take a look of the dataset.

In [ ]:
##### DO NOT CHANGE #####
from util import read_lines
train_de = read_lines('data/multi30k/train.de')
train_en = read_lines('data/multi30k/train.en')
valid_de = read_lines('data/multi30k/valid.de')
valid_en = read_lines('data/multi30k/valid.en')

print(f"German : {train_de[3]}")
print(f"English: {train_en[3]}")

##### DO NOT CHANGE #####

### Preprocessing


We need several pre-processing steps for the dataset, so that we can train our model on it. These steps include:
1. Get the vocabulary dictionaries for both German and English
2. Tokenize the sentences using the vocabulary dictionaries
3. Create PyTorch datasets and dataloaders

To do so, we will use the pre-defined tokenizer from the library `spacy` (https://spacy.io/).
It is already part of the uv project for this exercise (see the `pyproject.toml` if you are interested, which also contains the necessary dependencies for the German and English tokenizers).

The next cell builds the vocabulary as a combination of all distinct tokens in the dataset plus four special tokens:
1. `<unk>` for unknown words (words do not exist in spacy tokenizer)
2. `<pad>` for padding words
3. `<bos>` for the beginning of the sentence
4. `<eos>` for the end of the sentence
The vocabulary is just a Python dictionary mapping strings to integer indices (tokens).

The padding is used when the length of a sentence is shorter than the maximum length of sentences in the mini-batch. For example, if we have 'Hello world.' and 'Machine learning is very cool.' in one mini-batch, we need to pad the shorter sentence 'Hello world.' with three padding tokens `<pad>` as follows:

|    0    |    1    |    2     | 3  |    4    |    5    |    6    |    7    |
|:-------:|:-------:|:--------:|:--:|:-------:|:-------:|:-------:|:-------:|
| \<bos\> |  Hello  |  world   | .  | \<eos\> | \<pad\> | \<pad\> | \<pad\> |
| \<bos\> | Machine | learning | is |  very   |  cool   |    .    | \<eos\> |

In [ ]:
##### DO NOT CHANGE #####
from util import build_vocab, tokenize_de, tokenize_en
vocab_de = build_vocab(train_de, tokenize_de)
vocab_en = build_vocab(train_en, tokenize_en)

print(f"Index of {'Mannschaft'} in the German vocabulary dictionary is {vocab_de['Mannschaft']}")
print(f"Index of {'football'} in the English vocabulary dictionary is {vocab_en['football']}")

##### DO NOT CHANGE #####

To tokenize the sentences and decode it back, we can use the following python code:

In [ ]:
##### DO NOT CHANGE #####
# Sentences to tokens
def encode(text, vocab, tokenizer):
    return [vocab["<bos>"]] + [
        vocab[token] if token in vocab else vocab["<unk>"] for token in
        tokenizer(text)] + [vocab["<eos>"]]

# Tokens to sentences
def decode(tokens, vocab):
    inv_vocab = {v: k for k, v in vocab.items()}
    return " ".join([inv_vocab[token] for token in tokens if token not in (
        vocab["<bos>"], vocab["<eos>"], vocab["<pad>"])])

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# Let's see how the encode and decode functions work
original_german_sentence = train_de[3]
print(f"Original German: {original_german_sentence}")
encoded_german_sentence = encode(original_german_sentence, vocab_de, tokenize_de)
print(f"Tokenized German: {encoded_german_sentence} \n")
decoded_german_sentence = decode(encoded_german_sentence, vocab_de)
print(f"Decoded German: {decoded_german_sentence}")


##### DO NOT CHANGE #####

To train the German to English translator, we need to process each pair of German and English sentences in the dataset. We will use the following code to process the dataset.

In [ ]:
##### DO NOT CHANGE #####
# Encode the German sentence and its corresponding English sentence
def data_process(sentences_de, sentences_en):
    data = []
    for src_text, tgt_text in zip(sentences_de, sentences_en):
        src_tensor_ = torch.tensor(encode(src_text, vocab_de, tokenize_de),
                                   dtype=torch.long)
        tgt_tensor_ = torch.tensor(encode(tgt_text, vocab_en, tokenize_en),
                                   dtype=torch.long)
        data.append((src_tensor_, tgt_tensor_))
    return data


# Create the PyTorch dataset classes
class TranslationDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data

    def __getitem__(self, idx):
        return self.data[idx]

    def __len__(self):
        return len(self.data)
    
    
# Get the PyTorch datasets instances
train_dataset = TranslationDataset(data_process(train_de, train_en))
valid_dataset = TranslationDataset(data_process(valid_de, valid_en))

##### DO NOT CHANGE #####

With the datasets in hand, now we implement our dataloader. Each time, the dataloader will return a mini-batch of the dataset. As each sentence in the mini-batch has a different length, we need a collate function to add padding tokens at the end of the short sentences. We will use the following code to implement the dataloader.

In [ ]:
##### DO NOT CHANGE #####
from torch.utils.data import DataLoader

# Collate function to pad short sentences 
def collate_fn(batch):
    src_batch, tgt_batch = [], []
    for src_sample, tgt_sample in batch:
        src_batch.append(src_sample)
        tgt_batch.append(tgt_sample)
    src_batch = nn.utils.rnn.pad_sequence(src_batch, padding_value=vocab_de[
        "<pad>"]).transpose(0, 1) # The transpose ensures batch first view
    tgt_batch = nn.utils.rnn.pad_sequence(tgt_batch, padding_value=vocab_en[
        "<pad>"]).transpose(0, 1)
    return src_batch, tgt_batch


train_dataloader = DataLoader(TranslationDataset(train_dataset), batch_size=32,
                              shuffle=True, collate_fn=collate_fn)
valid_dataloader = DataLoader(TranslationDataset(valid_dataset), batch_size=32,
                              shuffle=True, collate_fn=collate_fn)

##### DO NOT CHANGE #####

### Model Implementation

OK, the data pre-processing part is finished. Now, let's take a look of the transformer architecture. The implementation has the following components:
1. TokenEmbedding
2. **Positional Encoding (your task)**
3. **Multi-Head Attention (your task)**
4. Feed Forward Neural Network
5. Layer Normalization
6. **Encoder Layer (your task)**
7. **Decoder Layer (your task)**
8. Encoder
9. Decoder
10. Transformer

Please note, for consistency to previous homework, we choose to use the **batch first dimensionality manner** in the implementation. This means the shape of a sequence data is always **(batch size, sequence length, ...)**, rather than the one commonly used in natural linear processing(NLP), as (sequence length, batch size, ...).

### 1.1) Token Embedding

We begin with the `TokenEmbedding`. It is used to convert the token (vocabulary) index to the corresponding token embedding, namely a high dimensional feature vector (512 in the current homework). 
The implementation uses the build-in pytorch embedding class and takes the size of the vocabulary (token) dictionary and the size of the embedding. 
We follow the original paper ["Attention is All You Need"](https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf) page 5, to scale the embedding by the square root of the embedding size.

In [ ]:
##### DO NOT CHANGE #####
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size: int, emb_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.emb_size = emb_size

    def forward(self, tokens: torch.Tensor):
        """
            tokens: [batch_size, seq_len]
            return: [batch_size, seq_len, emb_size]
        """
        token_embed = self.embedding(tokens.long()) * math.sqrt(self.emb_size)
        return token_embed

##### DO NOT CHANGE #####

### 1.2) Positional Encoding (4 Points)

Next, we need to implement the positional encoding. The positional encoding is used to add the information about the position of the tokens in the sentence, which is later added to the token embedding. It is calculated as follows:

$\text{PE}(t, 2l) = sin\left(\frac{t}{10000^{2l/d_{\text{model}}}}\right)  $

$\text{PE}(t, 2l+1) = cos\left(\frac{t}{10000^{2l/d_{\text{model}}}}\right)$

where $t$ is the positional index of the sequence and $l \in \{0, ..., d_{\text{model}} / 2 - 1\}$ is the index of the embedding. The embedding's dimension size is equivalent to the model dimension $d_{\text{model}}$ of the transformer.

In the next cell, pre-compute the values for the positional encoding depending on the token position, from zero (inclusive) up to a maximum length of `max_len` (exclusive).
Store the log-space values to `self.encoding` which is of shape `(max_len, emb_size)`, where `emb_size` equals $d_{\text{model}}$.
Make sure that `self.encoding[t, i]` equals $\text{PE}(t, i)$.

Tip: To write to every second element of a tensor `a` starting from the `k`th element, use `a[k::2]`.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, emb_size: int, max_len: int = 5000):        
        super().__init__()
        
        # Shape of the positional encoding: (max_len, emb_size)
        self.encoding = torch.zeros(max_len, emb_size)
        self.encoding.requires_grad = False  # No gradients needed

        # TODO Implement the positional encoding according to the description above.
        # YOUR CODE HERE
        raise NotImplementedError()
        
        # Add additional batch dimension in front, to (1, max_len, emb_size)
        self.encoding = self.encoding.unsqueeze(0) 

    def forward(self, token_embed: torch.Tensor) -> torch.Tensor:
        """
            token_embed: [batch_size, seq_len, emb_size]
            pos_enc: [batch_size, seq_len, emb_size]
        """
        seq_len = token_embed.size(1)
        pos_enc = token_embed + self.encoding[:, :seq_len, :].to(token_embed.device)
        return pos_enc

In [ ]:
##### DO NOT CHANGE #####
# ID: ex_1_2_tests - possible points: 4

# Some tests

test_pos_encoder = PositionalEncoding(512, 5000)

assert test_pos_encoder.encoding.shape == (1, 5000, 512), test_pos_encoder.encoding.shape
assert not (test_pos_encoder.encoding == 0.0).all()

test_embed = torch.ones((2, 10, 512))
test_pos_enc = test_pos_encoder(test_embed)
assert test_pos_enc is not None
assert test_pos_enc.shape == (2, 10, 512), test_pos_enc.shape


##### DO NOT CHANGE #####

Let's see what the positional encoding looks like. 
The next cell plots the magnitude of each embedding's dimension depending (x-axis) depending on the position in the sequence (y-axis).
We also offer a reference image `positional_encoding_reference.png` for you to compare and debug.

In [ ]:
##### DO NOT CHANGE #####
from util import show_positional_encoding
show_positional_encoding(PositionalEncoding)


##### DO NOT CHANGE #####

## 1.3) Multi-Head Attention (5 Points)

Next, we need to implement the scaled dot-product attention and the multi-head attention.

![MHA overview, reproduced from Attention is all you need, Vaswani et al., 2017](./attention.png)



The attention mechanism can learn a (soft) mapping between a query (Q) and a set of key (K) value (V) pairs, where the Q, K, V are linear projections from model features.

The formula for scaled dot-product attention is

$ \text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V.$

In our code, the shape of Q, K, V are:

- Q: `[batch_size, seq_len_q, d_k]`
- K: `[batch_size, seq_len_k, d_k]`
- V: `[batch_size, seq_len_k, d_v]`

where `d_k` and `d_v` can, in principle, be arbitrary sizes.
In practices, one typically chooses `d_k = d_v`, and sets `d_k` to `d_model / h`, where `h` is the number of attention heads (see below).

Note how Q has a different shape than K and V.
This is because Q is computed from a different sequence than K and V (cross-attention).
If all three features would have been computed from the same sequence, the operation would be called self-attention.

**Multi-Head Attention** Additionally, we can use multi-head attention (MHA), where we have multiple (Q, K, V) triplets, compute attention on each, and concatenate the result.
To not introduce any computational overhead, MHA usually operates in a feature space of reduced size $d_\text{model} / h$, given $h$ attention heads.
As such, we have $d_k = d_v = d_\text{model} / h$.

If we simply concatenate the features, information from different attention heads cannot mix before the subsequent incoming residual connection and normalization layer.
MHA therefore adds a linear out-projection $W^O$ after the attention operation:

$
\begin{align}
\text{MultiHead}(Q_{1, .., h}, K_{1, .., h}, V_{1, .., h}) &= \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O \\
\text{head}_i &= \text{Attention}(Q_i, K_i,, V_i)
\end{align}
$

Overall, each head needs three projection matrices $W^Q_i$, $W^K_i$, $W^V_i$ of shape $d_\text{model} \times d_\text{model} / h$, plus one shared out-projection matrix $W^O$ of shape $d_\text{model} \times d_\text{model}$.
While we could compute the per-head projections separately, we can also have three big projections $W^Q, W^K, W^V \in \mathbb{R}^{d_\text{model} \times d_\text{model}}$, giving us joint $Q, K, V$, which can then split each into $h$ per-head $Q_i, K_i, V_i$.
This can be significantly faster on modern GPUs, as it requires less separate kernel invocations.

**Masking**  Masking means setting the pre-softmax attention scores $\frac{QK^T}{\sqrt{d_k}}$ to a large negative number such as $-10^{9}$, so that the corresponding values get multiplied by zero and thus ignored. It is important to apply masking before the softmax, as otherwise the attention scores after the softmax wouldn't sum to one. 

There are two reasons for applying masking:
- Tokens might be padding, and we obviously don't want to attend to these tokens, as they carry no information and are only present during training.
- Self-Attention in a transformer decoder needs to be causal, i.e., tokens later in the sequence must not attend to earlier tokens. This is necessary because we later compute a next-token prediction loss for each token in the sequence. If attention wasn't causal, the model could use information from later tokens to predict earlier tokens, which would make autogressive prediction impossible.

**Your Task**
Implement scaled multi-head attention with dropout in the next cell. 
Some important details:
- The $Q, K, V$ features of shape `[B, T, h * d_k]` need to be reorganized into shape `[B, h, T, d_k]`. Use `torch.view` to split the feature dimension and `torch.transpose` to permute dimensions.
- For masking, apply the mask to all elements where `mask == 1`. Set these entries to `-1e9`. Use `torch.masked_fill` to selectively set entries to a fixed number wherever the given mask is `True`. Note that `merge_masks` may return `None`, in which you case you should not apply any masking.
- To avoid overfitting, apply attention dropout by passing the *post*-softmax scores through `self.dropout`.
- You do not need to re-assemble the per-head values and apply the out-projection. The provided code already does this.

In [ ]:
from util import merge_masks

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int = 512, n_head: int = 8, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_head == 0
        self.d_model = d_model
        self.n_head = n_head
        self.d_k = d_model // n_head
        self.q_net = nn.Linear(d_model, d_model)
        self.k_net = nn.Linear(d_model, d_model)
        self.v_net = nn.Linear(d_model, d_model)
        self.output_linear = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(p=dropout)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, q, k, v, causal_mask=None, padding_mask=None):
        """
            q: [batch_size, seq_len_q, d_model]
            k: [batch_size, seq_len_k, d_model]
            v: [batch_size, seq_len_k, d_model]
            causal_mask: [seq_len_q, seq_len_k]
            padding_mask: [batch_size, seq_len_k]
            return: [batch_size, seq_len_q, d_model]

            if self attention, q, k, v are the same, and seq_len_q = seq_len_k
            if cross attention, q != k = v, and seq_len_q != seq_len_k

        """

        batch_size = q.shape[0]

        # We need some special care to merge the masks. The mask_ij basically
        # tells if the i-th token in query is allowed to attend to the j-th
        # token in key.
        # The shape of mask is [batch_size, n_head, len_q, len_k] or any shape
        # that can be broadcasted to this shape
        mask = merge_masks(causal_mask, padding_mask, self.n_head)

        # Linear projections
        q, k, v = self.q_net(q), self.k_net(k), self.v_net(v)

        # TODO: Implement the multi-head attention
        # Consider the following steps:
        # 1. Split the Q, K, V into n_head by using torch.view
        # 2. Put the head axis after the batch dimension using torch.transpose
        # 3. Calculate the attention scores using Q, K, V
        # 4. Apply the mask to the attention scores by using torch.masked_fill
        # 5. Apply the softmax function to the attention scores
        # 6. Apply the dropout to the softmax scores to avoid overfitting
        # 7. Multiply the softmax scores with V

        # YOUR CODE HERE
        raise NotImplementedError()

        ####################################################################

        # Concatenate and multiply get processed by the output layer
        # [batch_size, n_head, T, d_k]
        # -> [batch_size, T, n_head, d_k]
        # -> [batch_size, T, n_head * d_k], where n_head * d_k = d_model
        attn = attn.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.output_linear(attn)



In [ ]:
##### DO NOT CHANGE #####
# ID: ex_1_3_tests - possible points: 5

# Some tests

torch.manual_seed(0)
test_attention = MultiHeadAttention(512, 8, 0.1)

test_q = torch.randn((2, 64, 512))
test_k = torch.randn((2, 64, 512))
test_v = test_k

# First without masking
test_out = test_attention(test_q, test_k, test_v, None, None)
assert test_out is not None
assert test_out.shape == (2, 64, 512), test_out.shape

# Now with masking
test_causal_mask = torch.triu(torch.ones((64, 64)), diagonal=1)
test_padding_mask = torch.ones((2, 64))

test_out_masked = test_attention(test_q, test_k, test_v, test_causal_mask, test_padding_mask)
assert test_out_masked is not None
assert test_out_masked.shape == (2, 64, 512), test_out_masked.shape



##### DO NOT CHANGE #####

### 1.4) Feed Forward Neural Network

A transformer block not only consists of the MHA, but also a two-layer feed forward neural network (MLP):

![A transformer block](./ff.png)

The MLP gets applied to all tokens of the sequence in parallel.
There is no information flow between tokens.
The MLP serves as the main "processing" power of the transformer.
The first layer maps tokens to a higher dimension $d_\text{ff} > d_\text{model}$, and the second layer maps them back down to the model dimension.
The next cell implements a simple two-layer MLP with ReLU activations.

In [ ]:
##### DO NOT CHANGE #####
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        x = self.dropout(torch.relu(self.linear1(x)))
        x = self.linear2(x)
        return x


##### DO NOT CHANGE #####

### 1.5) Layer Normalization

The last ingredient for a transformer block is layer normalization.
Compared to batch normalization, which normalizes each feature dimension across a batch, layer normalization normalizes the features of each individual batch member.
The operation is therefore independent of the other batch members, making training more stable and easier to parallelize across multiple GPUs.
Check out [https://www.pinecone.io/learn/batch-layer-normalization/]() for an in-depth explanation!

The next cell implements layer normalization.
Note that the the module has learnable parameters, that scale and shift features after normalization.

In [ ]:
##### DO NOT CHANGE #####
# Equivalent to torch.nn.LayerNorm(d_model)
class LayerNorm(nn.Module):
    def __init__(self, features: int, eps: float = 1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(features))
        self.beta = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta

##### DO NOT CHANGE #####

### 1.6) Encoder Block (4 Points)

We are now ready to implement the transformer blocks.
First, the encoder block.
These process the input sequence in parallel - for our German-to-English translation task, the encoder block processes the German sentence.

An encoder block consists of two sub-blocks (MHA and FF), both followed by dropout, wrapped with a residual connection, and a final LayerNorm. 
In code: The output of each sub-block is `LayerNorm(x + dropout(sublayer(x)))`, where `sublayer` is either MHA or the feed-forward NN (FF). 
Note the order of the incoming residual branch and the LayerNorm.

![An encoder block](./encoder_layer.jpg)

**Your Task** Implement the encoder block in the next cell. Use the provided `self.self_attn`, `self.ffn`, and `self.norm1`/`self.norm2`, `self.dropout1`/`self.dropout2` modules.

If you didn't manage to implement the attention layer, you can use the one by PyTorch; see the comments in the next cell for that. For the encoder block tests, it is only important that your attention layer produces outputs of the correct shape, not that the values themselves are correct.

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_head, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_head, dropout)

        # NOTE If you didn't manage to implement the MHA layer, remove the above line that uses your implementation 
        # NOTE and uncomment the next two lines to use the layer provided by PyTorch
        # self._torch_attention = torch.nn.MultiheadAttention(d_model, n_head, dropout, batch_first=True)
        # self.self_attn = lambda q, k, v, causal_mask=None, padding_mask=None: self._torch_attention(q, k, v, key_padding_mask=padding_mask, attn_mask=causal_mask)[0]

        self.ffn = FeedForward(d_model, d_ff, dropout)

        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, src, causal_mask=None, padding_mask=None):
        """
            src: [batch_size, seq_len, d_model]
            causal_mask: [seq_len, seq_len]
            padding_mask: [batch_size, seq_len]
            return: [batch_size, seq_len, d_model]
        """

        # TODO Implement the encoder layer with MHA, dropout, feed forward network, residual connections and layer normalization.
        # TODO Pass the provided causal_mask and padding_mask to the self-attention layer.
        # YOUR CODE HERE
        raise NotImplementedError()
        return src


In [ ]:
##### DO NOT CHANGE #####
# ID: ex_1_6_tests - possible points: 4

# Some tests

torch.manual_seed(0)
test_encoder_block = EncoderBlock(256, 8, 512)

# Test without masking
test_out = test_encoder_block(torch.ones((2, 64, 256)))
assert test_out is not None
assert test_out.shape == (2, 64, 256)

# Test with masking
test_out = test_encoder_block(torch.ones((2, 64, 256)), None, torch.ones((2, 64)))
assert test_out is not None
assert test_out.shape == (2, 64, 256)


##### DO NOT CHANGE #####

### 1.7) Decoder Block (6 Points)

Our transformer also needs decoder blocks.
This block is used to process the generated English sentence in our German-to-English translation task, while cross-attending to the encoder features.

Decoder blocks are similar to the encoder blocks, but with an additional cross-attention sub-block between the MHA and FF sub-blocks.
The sub-layers are again wrapped with residual connections and layer normalization, as in the encoder block.

![A decoder block](./decoder_layer.jpg)

**Your task** Implement the decoder block in the next cell. Use the provided `self.self_attn`, `self.cross_attn`, `self.ffn`, and `self.norm1`/`self.norm2`/`self.norm3`, `self.dropout1`/`self.dropout2`/`self.dropout3` modules.
The `memory` argument contains the encoder features, which are used for cross-attention. The `memory_mask`/`memory_padding_mask` are used to mask the encoder features, and the `tgt_mask`/`tgt_padding_mask` are used to mask the decoder features.
Pass them to the corresponding attention modules.

If you didn't manage to implement the attention layer, you can use the one by PyTorch; see the comments in the next cell for that. For the tests of the encoder block, it is only important that your attention layer produces outputs of the correct shape, not that the values themselves are correct.

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_head, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_head, dropout=dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_head, dropout=dropout)

        # NOTE If you didn't manage to implement the MHA layer, remove the above two lines that use your implementation 
        # NOTE and uncomment the next four lines to use the layer provided by PyTorch
        # self._torch_self_attention = torch.nn.MultiheadAttention(d_model, n_head, dropout, batch_first=True)
        # self.self_attn = lambda q, k, v, causal_mask=None, padding_mask=None: self._torch_self_attention(q, k, v, key_padding_mask=padding_mask, attn_mask=causal_mask)[0]
        # self._torch_cross_attention = torch.nn.MultiheadAttention(d_model, n_head, dropout, batch_first=True)
        # self.cross_attn = lambda q, k, v, causal_mask=None, padding_mask=None: self._torch_cross_attention(q, k, v, key_padding_mask=padding_mask, attn_mask=causal_mask)[0]

        self.ffn = FeedForward(d_model, d_ff, dropout)

        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
        self.norm3 = LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None,
                tgt_padding_mask=None, memory_padding_mask=None):
        """
            tgt: [batch_size, seq_len_tgt, d_model]
            memory: [batch_size, seq_len_src, d_model]
            tgt_mask: [seq_len_tgt, seq_len_tgt]
            memory_mask: [seq_len_tgt, seq_len_src]
            tgt_padding_mask: [batch_size, seq_len_tgt]
            memory_padding_mask: [batch_size, seq_len_src]
            return: [batch_size, seq_len, d_model]"""

        # TODO: Implement the decoder layer
        # Consider the following steps:
        # 1. Calculate the (masked) self-attention of the target
        # 2. Add the dropout, residual connection, and apply the layer normalization
        # 3. Calculate the (masked) cross-attention of the target with the memory
        # 4. Add the dropout, residual connection, and apply the layer normalization
        # 5. Apply the feed-forward neural network
        # 6. Add the dropout, residual connection, and apply the layer normalization

        # YOUR CODE HERE
        raise NotImplementedError()

        return tgt


In [ ]:
##### DO NOT CHANGE #####
# ID: ex_1_7_tests - possible points: 6

# Some tests

torch.manual_seed(0)
test_decoder_block = DecoderBlock(256, 8, 512)

# Test without masking
test_out = test_decoder_block(torch.ones((2, 64, 256)), torch.ones((2, 64, 256)))
assert test_out is not None
assert test_out.shape == (2, 64, 256)

# Test with masking
test_out = test_decoder_block(torch.ones((2, 64, 256)), torch.ones((2, 64, 256)), None, None, torch.ones((2, 64)), torch.ones((2, 64)))
assert test_out is not None
assert test_out.shape == (2, 64, 256)


##### DO NOT CHANGE #####

### 1.8) Encoder, Decoder, and Transformer

The next cell builds the encoder and decoder from the blocks you implemented above.
Note how both are simple stacks of the respective blocks.

After that, the transformer is simple to implement: It first passes the German sentence through the encoder to build a latent representation ("memory").
Then, the decoder processes the English sentence generated so far, while cross-attending to the encoder features.

In [ ]:
##### DO NOT CHANGE #####
class Encoder(nn.Module):
    def __init__(self, d_model, n_head, n_layers, d_ff, dropout=0.1):
        super().__init__()

        self.n_head = n_head
        self.layers = nn.ModuleList(
            [EncoderBlock(d_model, n_head, d_ff, dropout)
             for _ in range(n_layers)])

    def forward(self, src, mask, padding_mask):
        for layer in self.layers:
            src = layer(src, mask, padding_mask)
        return src


class Decoder(nn.Module):
    def __init__(self, d_model, n_head, n_layers, d_ff, dropout=0.1):
        super().__init__()

        self.layers = nn.ModuleList(
            [DecoderBlock(d_model, n_head, d_ff, dropout)
             for _ in range(n_layers)])

    def forward(self, tgt, memory, tgt_mask, memory_mask,
                tgt_padding_mask, memory_padding_mask):
        for layer in self.layers:
            tgt = layer(tgt, memory, tgt_mask, memory_mask,
                        tgt_padding_mask, memory_padding_mask)
        return tgt

class Transformer(nn.Module):
    def __init__(self, d_model, n_head,
                 num_encoder_layers, num_decoder_layers, d_ff):
        super().__init__()

        self.encoder = Encoder(d_model, n_head, num_encoder_layers, d_ff)
        self.decoder = Decoder(d_model, n_head, num_decoder_layers, d_ff)

    def forward(self, src, tgt, src_mask, tgt_mask, memory_mask,
                src_padding_mask, tgt_padding_mask, memory_padding_mask
                ):
        memory = self.encoder(src, src_mask, src_padding_mask)
        output = self.decoder(tgt, memory, tgt_mask, memory_mask,
                              tgt_padding_mask, memory_padding_mask)
        return output

##### DO NOT CHANGE #####

### 1.9) Putting Everything Together

Finally, we can create the transformer translator by combining token embeddings, positional encodings, the transformer itself, and a final linear layer ("generator") that maps from the model dimension to the vocabulary size.

If you didn't manage to implement the transformer, you can uncomment the lines in the cell below where PyTorch's transformer implementation (`torch.nn.Transformer`) is used.
Note that this implementation is not identical to ours, and may not produce identical loss and predictions, so don't use it to verify that your implementation is correct.

In [ ]:
##### DO NOT CHANGE #####
class TransformerTranslator(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, n_head,
                 num_encoder_layers, num_decoder_layers, d_ff):
        super().__init__()

        self.transformer = Transformer(d_model=d_model, n_head=n_head,
                          num_encoder_layers=num_encoder_layers,
                          num_decoder_layers=num_decoder_layers,
                          d_ff=d_ff)
        
        # NOTE: If you cannot make your model work, here is PyTorch's built-in Transformer
        #self.transformer = torch.nn.Transformer(d_model=emb_size, nhead=nhead,
        #                  num_encoder_layers=num_encoder_layers,
        #                  num_decoder_layers=num_decoder_layers,
        #                  dim_feedforward=d_ff)
        
        self.generator = nn.Linear(d_model, tgt_vocab_size)
        self.src_tok_emb = TokenEmbedding(src_vocab_size, d_model)
        self.tgt_tok_emb = TokenEmbedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model)

    def forward(self, src, tgt, src_mask, tgt_mask, src_padding_mask,
                tgt_padding_mask, memory_padding_mask):

        # Get token embeding and positional encoding
        src_emb = self.positional_encoding(self.src_tok_emb(src))
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(tgt))

        # Get model predictions
        outs = self.transformer(src=src_emb, tgt=tgt_emb, src_mask=src_mask,
                                tgt_mask=tgt_mask, memory_mask=None,
                                src_padding_mask=src_padding_mask,
                                tgt_padding_mask=tgt_padding_mask,
                                memory_padding_mask=memory_padding_mask)

        # Pass the final features through the generator layer to map to final pre-softmax token probabilities
        return self.generator(outs)

##### DO NOT CHANGE #####

### 1.10) Training

The next cells contain the training code.
The first contains hyperparameters that you are free to change.
Ignore the second cell.

The third defines the loss function that iterates once over a dataloader, computes the loss per batch, and updates the model if an optimizer is provided.

The fourth cell contains the actual training loop.
Running it will start training.
For reference: On an NVIDIA RTX 5090, one epoch takes approximately 18 seconds, and the entire training takes approximately 3 minutes.
If you need to train on the CPU, one epoch will take multiple minutes.

A GPU with >= 2GB VRAM should be sufficient.

In [ ]:
# You are free to change this cell to reduce the number of training epochs or change the device that PyTorch uses

# 'cuda' for GPU (NVIDIA only), 'cpu' for cpu, 'mps' for Macs
# Your TA has no Mac to verify that mps works, so you may need to try it yourself... 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') 

N_EPOCHS = 10

In [ ]:
##### DO NOT CHANGE #####
# ID: dummy_test_pls_ignore - possible points: 0

# Ignore this cell; it disables training during notebook grading. It does not actually run any tests, and you can't receive any points.

TRAIN = True


##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
from tqdm import tqdm
from util import create_mask

# Note how we exclude the padding tokens in loss calculation. The model is free to predict whatever it wants on these tokens, as they do not exist during model evaluation.
criterion = nn.CrossEntropyLoss(ignore_index=vocab_en['<pad>'])

def compute_loss(model, data_loader, criterion, optimizer=None):
    if optimizer is not None:
        # train mode
        model.train() # train mode: enable dropout
    else:
        # eval mode
        model.eval()  # eval mode: disable dropout

    total_loss = 0
    for src, tgt in tqdm(data_loader):
        src, tgt = src.to(device), tgt.to(device)

        # We skip the last token of the target sentence, as it would never be input to our model - as soon as it outputs <eos>, we stop generation.
        tgt_input = tgt[:, :-1]

        # We need to create masks for correct attention behaviour
        src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(
            src, tgt_input, vocab_de, vocab_en, device)

        logits = model(src, tgt_input, src_mask, tgt_mask, src_padding_mask,
                       tgt_padding_mask, src_padding_mask)

        tgt_out = tgt[:, 1:]
        loss = criterion(logits.reshape(-1, logits.shape[-1]),
                         tgt_out.reshape(-1))

        if optimizer is not None:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        total_loss += loss.item()

    return total_loss / len(data_loader)

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
import os
import torch.optim as optim

# create output directory for saved models
os.makedirs("./model", exist_ok=True)

torch.manual_seed(0)

# The hyperparameters below are in no way optimized
# 6 layers for the encoder and decoder is quite small
# We set d_model = d_ff to reduce the VRAM requirements for training
model = TransformerTranslator(len(vocab_de), len(vocab_en), 512, 8, 6, 6, 512).to(
    device)

optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-5)

if TRAIN:
    best_valid_loss = float('inf')
    for epoch in range(N_EPOCHS):
        train_loss = compute_loss(model, train_dataloader, criterion, optimizer)

        with torch.no_grad():
            valid_loss = compute_loss(model, valid_dataloader, criterion, optimizer=None)

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss

            torch.save(model, 'model/model_best.pth')

        print(f'Epoch: {epoch + 1}, Loss: {train_loss}, Val_loss: {valid_loss}')
    torch.save(model, 'model/model_last.pth')
    print("Training completed!")

##### DO NOT CHANGE #####

### 1.11) Inference

Finally, we can evaluate the model.
The next cell defines the autoregressive decoding code: We first encode the input sentence, and then iteratively predict one token and append it to the output sequence, starting with the `<bos>` token. We always take the most likely token as the next token, using a hard `argmax` operation. One could also sample from the token distribution, potentially after flattening or sharpening it with a temperature parameter.

In [ ]:
##### DO NOT CHANGE #####
from util import generate_square_subsequent_mask

# Function to translate sentences
def translate_sentence(sentence, model, vocab_de, vocab_en, tokenizer_de,
                       max_length=50):
    model.eval()
    src = torch.tensor(encode(sentence, vocab_de, tokenizer_de),
                       dtype=torch.long).unsqueeze(0).to(device)
    
    # The translation starts with token <bos>
    tgt = torch.tensor([vocab_en["<bos>"]], dtype=torch.long).unsqueeze(0).to(
        device)

    src_mask = torch.zeros((src.shape[1], src.shape[1]), device=device).type(
        torch.bool)

    memory = model.transformer.encoder(
        model.positional_encoding(model.src_tok_emb(src)), src_mask, None)

    for i in range(max_length):        
        tgt_mask = generate_square_subsequent_mask(tgt.size(1), device)

        out = model.transformer.decoder(
            model.positional_encoding(model.tgt_tok_emb(tgt)), memory, tgt_mask, None, None, None)
        out = model.generator(out[:, -1])
        prob = out.softmax(dim=-1)
        next_word = prob.argmax(dim=-1).item()

        # We append the predicted word to the output sentence
        tgt = torch.cat([tgt, torch.tensor([[next_word]], device=device)],
                        dim=1)
        
        # The translation ends with token <eos>
        if next_word == vocab_en["<eos>"]:
            break

    return decode(tgt.squeeze().tolist(), vocab_en)

##### DO NOT CHANGE #####

As the next cell shows, the model performs relatively well on sentences that are close to those in the training data (though it is not perfect, see e.g. the last sentence):

In [ ]:
##### DO NOT CHANGE #####
# Given a list of German sentences
german_list = [
    "Eine junge Frau sitzt auf einer Mauer und schaut aufs Meer.",
    "Mehrere Kinder rennen über einen Spielplatz.",
    "Ein älterer Mann geht mit seinem Hund im Park spazieren.",
    "Die Familie sitzt auf der Veranda und isst Frühstück.",
    "Eine Gruppe von Touristen besichtigt eine historische Burg.",
    "Ein Mädchen pflückt Blumen in einem blühenden Garten.",
    "Der Musiker spielt ein Lied auf seiner Geige in der Fußgängerzone.",
    "Die Katze liegt faul auf dem Fensterbrett und sonnt sich.",
    "Das Paar fährt mit dem Fahrrad durch die malerische Landschaft.",
    "Der Koch bereitet ein festliches Abendessen in der Küche vor.",
    "Eine Menschenmenge schaut einem Straßenkünstler zu.",
    "Die Kinder bauen eine Sandburg am Strand.",
    "Der Maler arbeitet an einem neuen Kunstwerk in seinem Atelier.",
    "Eine Frau liest ein Buch unter einem schattigen Baum.",
    "Der Hund jagt einen Ball über die Wiese.",
    "Eine Gruppe von Freunden zeltet in den Bergen.",
    "Der Fotograf macht Bilder von der Skyline der Stadt.",
    "Die Tänzer üben eine Choreografie im Studio.",
    "Der Bauer fährt mit dem Traktor über sein Feld.",
    "Die Schülerin macht ihre Hausaufgaben am Schreibtisch."
]

if TRAIN:

    # Load the model best or last
    # use_model = torch.load('model/model_best.pth', weights_only=False)
    use_model = torch.load("model/model_last.pth", weights_only=False)

    torch.manual_seed(0)

    # Translate each sentence in the list
    for german_sentence in german_list:
        translation = translate_sentence(german_sentence, use_model, vocab_de, vocab_en,
                                        tokenize_de)
        print(f"German: {german_sentence}")
        print(f"English: {translation}\n")

##### DO NOT CHANGE #####

However, for harder sentences, the performance is quite bad:

In [ ]:
##### DO NOT CHANGE #####
hard_german_list = ["Die deutsche Fußball-Nationalmannschaft ist mit einem überzeugenden Sieg in die Fußball-Weltmeisterschaft gestartet.",
                    "Im Sechzehntelfinale schied das Team von Bundestrainer Julian Nagelsmann allerdings gegen Paraguay aus."]

if TRAIN:
    for german_sentence in hard_german_list:
        translation = translate_sentence(german_sentence, use_model, vocab_de, vocab_en,
                                        tokenize_de)
        print(f"German: {german_sentence}")
        print(f"English: {translation}\n")

##### DO NOT CHANGE #####

### 1.12) Further Improvements (1P)

Based on your knowledge from the lecture, which of the following changes to the model and training setup is the **least** likely to improve the translation quality?

- a) More training data
- b) More decoder blocks
- c) A larger $d_\text{model}$
- d) Switching to a recurrent architecture

In the next cell, write your answer as a Python string containing one lowercase letter in `answer_1_12`.

In [ ]:
answer_1_12 = None
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: ex_1_12_tests - possible points: 1

assert answer_1_12 in ("a", "b", "c", "d")


##### DO NOT CHANGE #####

# 2) Divergences (3 Points)

You don't need to implement anything here; this is just a sequence of multiple-choice questions about the Kullback-Leibler (KL) divergence.

Th KL divergence between distributions $p_1$ and $p_2$ is defined as
$
\begin{align}
\text{KL}[p_1~||~p_2] = \int p_1(x) \log \frac{p_1(x)}{p_2(x)} dx = \mathbb{E}_{x \sim p_1}\left[\log p_1(x) - \log p_2(x) \right]
\end{align}
$

In practice, there is usually a ground-truth distribution $p$ and we want to find an approximation $q$.
$p$ can be arbitrarily complex, and $q$ usually follows some parametric model class, for example, a Gaussian where the mean is given by a neural network, or a diffusion model.
Since the KL divergence is not symmetric, we can choose between the *forward KL* $\text{KL}[p~||~q]$ and the reverse KL $\text{KL}[q~||~p]$.

### 2.1) Forward vs. Reverse KL - Multi-Modal Targets (1 Point)

In the two plots below, we have a multi-modal target density $p$ and trained a unimodal Gaussian $q$ to minimize either the forward KL or the reverse KL.

![KL Quiz](./kl_quiz.jpg)

Which one is which?

- a) Left is forward KL, right is reverse KL
- b) Left is reverse KL, right is forward KL

In the next cell, write your answer as a Python string containing one lowercase letter in `answer_2_1`.

In [ ]:
answer_2_1 = None
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: ex_2_1_test - possible points: 1

assert answer_2_1 in ("a", "b")


##### DO NOT CHANGE #####

### 2.2) Forward vs. Reverse KL - Data Fitting (1 Point)

You are given a dataset of samples $\{x_i ~|~ x_i\sim p\}$ from a target density $p$, but you have no access to $p$ itself.
To fit a probabilistic model that approximates $p$, which KL divergence are possible to use, given the information about $p$ that you have?
- a) Forward KL
- b) Reverse KL
- c) Both

In the next cell, write your answer as a Python string containing one lowercase letter in `answer_2_2`.

In [ ]:
answer_2_2 = None
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: ex_2_2_test - possible points: 1

assert answer_2_2 in ("a", "b", "c")


##### DO NOT CHANGE #####

### 2.3) Forward vs. Reverse KL - Entropy (1 Point)

We can rewrite both KL divergences as a sum of an expected log-likelihood and an entropy term (please verify this on your own).
Which type of KL divergence depends on the entropy of the fitted density, $\mathcal{H}[q]$?
- a) Forward KL
- b) Reverse KL

In the next cell, write your answer as a Python string containing one lowercase letter in `answer_2_3`.

In [ ]:
answer_2_3 = None
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: ex_2_3_test - possible points: 1

assert answer_2_3 in ("a", "b")


##### DO NOT CHANGE #####